# Boarding pass generation (instructor prep, not a student module)

This notebook renders the boarding pass used as the worked example in M02.03, plus its ground truth:

- `data/boarding_pass/boarding_pass.png`
- `data/boarding_pass/ground_truth.json`

It is kept apart from `generate_data.ipynb` so regenerating it never touches the invoices or figures. Rerunning it overwrites both files; the output is deterministic (fixed barcode seed).

## Setup

In [ ]:
import json
import platform
import random
from pathlib import Path

from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

from agentic_workshop.config import DATA_DIR

BOARDING_PASS_DIR = DATA_DIR / "boarding_pass"

_FONTS = {  # (regular, bold) per operating system, as in generate_data.ipynb
    "Darwin": (
        "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf",
    ),
    "Windows": ("C:/Windows/Fonts/arial.ttf", "C:/Windows/Fonts/arialbd.ttf"),
    "Linux": (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    ),
}


def font(size, bold=False):
    path = _FONTS.get(platform.system(), ("", ""))[bold]
    if path and Path(path).exists():
        return ImageFont.truetype(path, size)
    return ImageFont.load_default(size=size)  # Pillow >= 10.1

In [ ]:
LABEL = font(14, bold=True)
VALUE = font(24, bold=True)
VALUE_SMALL = font(18, bold=True)
TITLE = font(28, bold=True)

RED = (226, 0, 26)
GREY = (110, 110, 110)
BLACK = (20, 20, 20)

## What is printed on the pass, and the expected extraction

The seat (`23A`) is one printed field that the schema splits into a row (`int`) and a letter (`str`); the gate and sequence numbers are other numbers the model must not confuse with it.

The ground truth follows the schema of M02.03's `BoardingPass`.

In [ ]:
PASS = dict(
    passenger="MEIER / SOPHIE",
    origin_city="GENEVA",
    origin_code="GVA",
    destination_city="ZURICH",
    destination_code="ZRH",
    flight="LX 2805",
    date="14NOV26",
    departure_time="07:10",
    boarding_time="06:40",
    travel_class="ECONOMY",
    seat="23A",
    gate="A12",
    sequence="042",
    cabin_bag="1 x 8 kg INCLUDED",
    checked_bag="NOT INCLUDED",
)

GROUND_TRUTH = dict(
    passenger_name="Sophie Meier",
    origin="Geneva",
    destination="Zurich",
    departure_date="2026-11-14",
    seat_row=23,
    seat_letter="A",
    cabin_bag_allowed=True,
    travel_class="Economy",
)

## Render the boarding pass

In [ ]:
def render_boarding_pass(p, rng):
    W, H = 1000, 420
    STUB_X = 720
    img = Image.new("RGB", (W, H), "white")
    d = ImageDraw.Draw(img)
    d.rectangle([0, 0, W - 1, H - 1], outline=(200, 200, 200), width=2)

    # Header band
    d.rectangle([0, 0, W, 64], fill=RED)
    d.text((30, 16), "SWISS", font=TITLE, fill="white")
    d.text((160, 22), "BOARDING PASS", font=VALUE_SMALL, fill="white")
    d.text((STUB_X + 30, 22), "BOARDING PASS", font=VALUE_SMALL, fill="white")

    # Perforation between the main part and the stub
    for y in range(72, H - 8, 14):
        d.line([(STUB_X, y), (STUB_X, y + 7)], fill=(170, 170, 170), width=2)

    def field(label, value, x, y, value_font=VALUE):
        d.text((x, y), label, font=LABEL, fill=GREY)
        d.text((x, y + 18), value, font=value_font, fill=BLACK)

    # Main part
    field("NAME OF PASSENGER", p["passenger"], 30, 84)
    field("FROM", f"{p['origin_city']} {p['origin_code']}", 30, 144)
    field("TO", f"{p['destination_city']} {p['destination_code']}", 330, 144)
    for i, (label, key) in enumerate([
        ("FLIGHT", "flight"), ("DATE", "date"), ("DEPARTURE", "departure_time"), ("BOARDING", "boarding_time"),
    ]):
        field(label, p[key], 30 + i * 165, 204)
    for i, (label, key) in enumerate([
        ("CLASS", "travel_class"), ("SEAT", "seat"), ("GATE", "gate"), ("SEQ", "sequence"),
    ]):
        field(label, p[key], 30 + i * 165, 264)
    field("CABIN BAG", p["cabin_bag"], 30, 324, VALUE_SMALL)
    field("CHECKED BAG", p["checked_bag"], 330, 324, VALUE_SMALL)

    # A decorative barcode (random bars, not an encoding of anything)
    x = 30
    while x < 680:
        width = rng.choice([2, 2, 3, 4])
        d.rectangle([x, 382, x + width - 1, 406], fill=BLACK)
        x += width + rng.choice([2, 3, 4])

    # Stub
    sx = STUB_X + 30
    field("NAME", p["passenger"], sx, 84, VALUE_SMALL)
    field("FROM / TO", f"{p['origin_code']} - {p['destination_code']}", sx, 144, VALUE_SMALL)
    field("FLIGHT", p["flight"], sx, 204, VALUE_SMALL)
    field("DATE", p["date"], sx + 120, 204, VALUE_SMALL)
    field("SEAT", p["seat"], sx, 264, VALUE_SMALL)
    field("CLASS", p["travel_class"], sx + 120, 264, VALUE_SMALL)

    return img

## Save the image and the ground truth

In [ ]:
BOARDING_PASS_DIR.mkdir(parents=True, exist_ok=True)
img = render_boarding_pass(PASS, random.Random(42))  # deterministic barcode
img.save(BOARDING_PASS_DIR / "boarding_pass.png")
(BOARDING_PASS_DIR / "ground_truth.json").write_text(
    json.dumps(GROUND_TRUTH, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Wrote boarding_pass.png and ground_truth.json to {BOARDING_PASS_DIR}")

### Sanity check: preview the boarding pass

In [ ]:
display(Image.open(BOARDING_PASS_DIR / "boarding_pass.png"))